# **1. Loading the datasets**

In [ ]:
%pip install pandas
%pip install seaborn matplotlib
%pip install catboost
%pip install scikit-learn lightgbm
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train_url = "https://raw.githubusercontent.com/Kriti0613/Heavy-Equipment-Selling-Price-Prediction/main/data/train.csv"
train = pd.read_csv(train_url)

test_url = "https://raw.githubusercontent.com/Kriti0613/Heavy-Equipment-Selling-Price-Prediction/main/data/test.csv"
test = pd.read_csv(test_url, low_memory=False)

In [ ]:
train.head()

In [ ]:
test.head()

# **2.** **Exploratory Data Analysis (EDA)**
## 2.1 Identifying the datatypes and dataset information

In [ ]:
train.info()

In [ ]:
test.info()

## 2.2 Performing descriptive statistical analysis on the dataset

In [ ]:
train.describe()

In [ ]:
test.describe()

## 2.3 Identifying missing values (if any)

In [ ]:
train.isna().sum()

In [ ]:
missing = train.isna().mean().sort_values(ascending=False)

missing = missing[missing>0]


import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
missing.plot(kind='bar')

plt.title("Percentage of Missing Values")
plt.ylabel("Missing Percentage")
plt.xlabel("Features")
plt.xticks(rotation=90)
plt.show()

***Observation:*** *Several variables contain missing values, with OperationalHoursMeter, UtilizationTier, and a few categorical attributes having the highest proportion of missing entries. However, no features contains missing values to the extent it becomes unusable.*

***Inference:*** *Instead of dropping these columns and losing potentially valueable information, appropriate imputation techniques will be applied during preprocessing the data.*

In [ ]:
test.isna().sum()

In [ ]:
missing = test.isna().mean().sort_values(ascending=False)

missing = missing[missing>0]

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
missing.plot(kind='bar')

plt.title("Percentage of Missing Values")
plt.ylabel("Missing Percentage")
plt.xlabel("Features")
plt.xticks(rotation=90)
plt.show()

## 2.4 Identifying duplicates (if any)

In [ ]:
print("Duplicates in Train Dataset: ", train.duplicated().sum())
print("Duplicates in Test Dataset: ", test.duplicated().sum())

## 2.5 Exploring Target Variable

In [ ]:
train['TargetValue'].describe()

In [ ]:
train['TargetValue'].skew()

In [ ]:
sns.histplot(train['TargetValue'], bins=50, kde=True)
plt.show()

***Observation:*** *The selling price (TargetValue) is positivelu skewed, with a large number of machines sold at lower prices and relatively fewer extremely expensive machines.*

***Inference:*** *Since the competition uses RMSLE as the evaluation metric, a log1p() transformation is applied to reduce skewness, stabilise variance, and reduce the influence of high-value observations during model training.*

# **3.** **Data Visualizations**

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(train['ManufactureYear'], bins=40)

plt.title('Distribution of Manufacture Year')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(train.loc[train['ManufactureYear']>1900, 'ManufactureYear'], bins=30, kde=True)

plt.title('Distribution of Manufacture Year')
plt.xlabel('Mfg Year')
plt.ylabel("Count")
plt.show()

## 3.1 Feature engineering to extract meaningful datapoints

In [ ]:
def extract_date(df):
    df = df.copy()
    df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
    df['TransactionYear'] = df['TransactionDate'].dt.year
    df['TransactionMonth'] = df['TransactionDate'].dt.month
    df['TransactionQuarter'] = df['TransactionDate'].dt.quarter
    return df

In [ ]:
train = extract_date(train)
test = extract_date(test)

In [ ]:
train['AssetAge'] = (train['TransactionYear'] - train['ManufactureYear'])
test['AssetAge'] = (test['TransactionYear'] - test['ManufactureYear'])

In [ ]:
sns.boxplot(x=pd.cut(train['AssetAge'],
                     bins = [0, 5, 10, 20, 40, 100]),
               y=train['TargetValue'])

***Observation:*** *Machines with lower AssetAge generally command higher sale prices, while older machinery tends to sell for lesser prices.*

***Inference:*** *AssetAge captures depreciation more effectively than manufacture year alone and is therefore included as an engineered feature.*

In [ ]:
sns.histplot(train['OperationalHoursMeter'].dropna(), bins=50)

In [ ]:
hours = train['OperationalHoursMeter'].dropna()

plt.figure(figsize=(10,5))
sns.histplot(hours[hours <= hours.quantile(0.99)],
            bins=50,
            kde=True)

plt.title("Distribution of OperationalHoursMeters (99th percentile)")
plt.xlabel("Operational Hours")
plt.ylabel("Count")
plt.show()

***Observation:*** *The distribution of operational hours is highly right-skewed. Most machines have relatively low recorded operating hours, while a small number have extremely high values.*
***Inference:*** *To better visualize the typical distribution, the histogram is restricted to the 99th percentile. The presence of outliers suggests that the median is a more appropriate imputation statistics than the mean for this feature.*

In [ ]:
train["OperationalHoursMeter"].describe(percentiles=[0.90, 0.95, 0.99])

In [ ]:
sns.scatterplot(x=hours[hours <= hours.quantile(0.99)], y=train['TargetValue'], alpha=0.2)

***Observation:*** *Machines with lower operational hours generally tend to have higher resale values, although the relationship is not perfectly linear.*

***Inference:*** *Operational hours provide useful information regarding machine usage and condition, making this an important predictive feature.*

In [ ]:
train['RegionCode'].value_counts().head(10).plot(kind='bar')

***Observation:*** *Machinery sales are concentrated in a few regions such as Florida, Texas, California, etc. While several other regions contribute realatively fewer transactions.*

***Inference:*** *Regional differences may reflect local market demand, and pricing behaviour, therefore, RegionCode is retained as a categorical predictor.*

## 3.2 Feature engineering to clean the data

In [ ]:
def clean_features(df):
    df = df.copy()

    df['HasOperationalHours'] = (df['OperationalHoursMeter'].notna().astype(int))

    df['HasVariantModifier'] = (df['Spec_VariantModifier'].notna().astype(int))

    return df

In [ ]:
train = clean_features(train)
test = clean_features(test)

In [ ]:
train['DescriptorLength'] = (train['Spec_FullDescriptor'].fillna('').astype(str).str.len())
test['DescriptorLength'] = (test['Spec_FullDescriptor'].fillna('').astype(str).str.len())

In [ ]:
train.drop(columns=['TransactionDate'], inplace=True)
test.drop(columns=['TransactionDate'], inplace=True)

In [ ]:
numeric = train.select_dtypes(include=np.number)

corr = numeric.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap='coolwarm', center=0)

***Observation:*** *Most numerical variables exhibit relatively weaker linear correlations with target variable, indicating that the selling price is influenced by multiple interacting factors rather than a single dominant predictor.Strong correlations observed between engineered features(eg, Transaction Quarter, Manufacture Year and AssetAge) are expected because one feature is derived from the other.*

***Inference:*** *The weak pairwise correlations suggest that ensemble tree-based models such as RandomForest and LightGBM are appopriate, as they can effectively capture complex, non-linear relationships that simple linear models may fail to model.*

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(data=train,
           x='UtilizationTier',
           y='TargetValue')

plt.title("Target Value across Utilization Tier")
plt.xlabel("UtilizationTier")
plt.ylabel('Target Value')

plt.show()

***Observation:*** *The resale price varies across different utilization tiers, indicating that machinery usage patterns influence market value.*

***Inference:*** *Since different utilization levels exhibit different price distributions, UtilizationTier is retained as a categorical predictor and encoded during preprocessing.*

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(train['DescriptorLength'],
            bins=30,
            kde=True)

plt.title("Distribution of Descriptor Length")
plt.xlabel("Descriptor Length")
plt.ylabel("Count")

plt.show()

***Observation:*** *Most machinery descriptors fall withing a moderate range of descriptor lengths, while only a small number contain exceptionally long specifications.*

***Inference:*** *The descriptor length serves as a simple proxy for the amount of technical info available about a machine. More detailed specs may indicate specialized equipment, potentially influencing resale value. Therefore DescriptorLength is included as an engineered feature.*

## **3.3 Key Insights from EDA**

The exploratory analysis revealed several patterns that guided the feature engineering process:
1. The target variable is positively skewed, motivating the use of log transformation before model training.
2. Older machinery usually exhibits lower resale prices, leading to the creation of the AssetAge feature.
3. Opeartional Hours are highly skewed and contain extreme outliers, making median imputation more appropriate than the mean imputation strategy.
4. Difference in resale prices across utilization tiers suggest that machine usage characteristics influence market value.
5. The varying lengths of machine descriptions indicate differing levels of specification detail, motivating the creation of the DescriptorLength feature.

Based on these observations, new features were engineered to better represent machine characteristics and improve predictive performance.

# **4. Data Preprocessing**

In [ ]:
train['TargetValue'].describe()

In [ ]:
train['TargetValue'].skew()

In [ ]:
X = train.drop(columns=['TargetValue'])
y = np.log1p(train['TargetValue'])

X_test = test.copy()

## 4.1 Imputing the data

In [ ]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print(f"Categorical columns: {len(cat_cols)}")
print(f"Numerical columns: {len(num_cols)}")

In [ ]:
from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, random_state=42, test_size = 0.2
)

X_train.shape, X_val.shape

In [ ]:
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_val[num_cols] = num_imputer.transform(X_val[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

In [ ]:
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_val[cat_cols] = cat_imputer.transform(X_val[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

## 4.2 Encoding the data

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([
        X_train[col],
        X_val[col],
        X_test[col]
    ], axis=0).astype(str)
    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    encoders[col] = le

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print("\nRemaining Missing values:")
print(X_train.isna().sum().sum())
print(X_val.isna().sum().sum())
print(X_test.isna().sum().sum())

# **5. Model Building**

## 5.1 First Model: Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_log_error

dt_model = DecisionTreeRegressor(max_depth=15,
                                min_samples_split=20,
                                random_state=42)

dt_model.fit(X_train, y_train)
dt_pred_log = dt_model.predict(X_val)
dt_pred = np.expm1(dt_pred_log)
y_val_actual = np.expm1(y_val)
dt_rmsle = root_mean_squared_log_error(y_val_actual, dt_pred)
print(f"Decision Tree Validation RMSLE: {dt_rmsle:.4f}")

## 5.2 Second Model: CatBoostRegressor

In [ ]:

from catboost import CatBoostRegressor

cat_features = X_train.select_dtypes(include='object').columns.tolist()
cat_model = CatBoostRegressor(iterations=500,
                             learning_rate=0.05,
                             depth=8,
                             loss_function='RMSE',
                             eval_metric='RMSE',
                             random_seed=42,
                             verbose=100)

cat_model.fit(X_train, y_train, cat_features=cat_features)
cat_pred_log = cat_model.predict(X_val)
cat_pred = np.expm1(cat_pred_log)
cat_rmsle=root_mean_squared_log_error(y_val_actual, cat_pred)
print(f"CatBoost Validation RMSLE: {cat_rmsle:.5f}")

## 5.3 Third Model: LightGBMRegressor

In [ ]:
import lightgbm as lbg

lbg_model = lbg.LGBMRegressor(
    objective="regression",
    metric="rmse",
    n_estimators = 12000,
    random_state=42,
    n_jobs=-1
)

lbg_model.fit(X_train, y_train)
lbg_pred_log = lbg_model.predict(X_val)
lbg_pred = np.expm1(lbg_pred_log)
y_val_actual = np.expm1(y_val)
lbg_rmsle = root_mean_squared_log_error(y_val_actual, lbg_pred)
print(f"LightGBM Regressor Validation RMSLE: {lbg_rmsle:.4f}")

# **6. Hyperparameter Tuning the third model - LightGBMRegressor**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "num_leaves": [127, 255],
    "learning_rate": [0.02, 0.03],
    "min_child_samples": [5, 10]
}

base_model = lbg.LGBMRegressor(
    metric="rmse",
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(estimator=base_model,
                                  param_distributions=param_dist,
                                  n_iter = 3,
                                  scoring="neg_root_mean_squared_error",
                                  cv=3,
                                  random_state=42,
                                  verbose=1,
                                  n_jobs=-1)

random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)

best_params = random_search.best_params_



In [ ]:
model = lbg.LGBMRegressor(objective="regression",
                          metric='rmse',
                          n_estimators=12000,
                          learning_rate=0.02,
                          num_leaves=255,
                          max_depth=-1,
                          min_child_samples=5,
                          subsample=0.8,
                          colsample_bytree=0.8,
                          reg_alpha=0.1,
                          reg_lambda=0.1,
                          random_state=42,
                          n_jobs=-1)

model.fit(X_train, y_train,
         eval_set=[(X_val, y_val)],
         eval_metric="rmse",
         callbacks=[
             lbg.early_stopping(300),
             lbg.log_evaluation(100)
         ])

In [ ]:
val_pred_log=model.predict(X_val)

In [ ]:
val_pred = np.expm1(val_pred_log)
y_val_actual =np.expm1(y_val)

In [ ]:
from sklearn.metrics import mean_squared_log_error
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

In [ ]:
score = rmsle(y_val_actual, val_pred)
print(f"Validation RMSLE: {score:.5f}")

In [ ]:
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)

test_pred = np.maximum(test_pred, 0)

# **7. Comparing the models**

In [ ]:
comparison = pd.DataFrame({
    "Model" : ["Decision Tree", "CatBoost", "LightGBM"],
    "Validation_RMSLE": [dt_rmsle, cat_rmsle, score]
})

comparison = comparison.sort_values("Validation_RMSLE")
comparison

## 7.1 Visualizing the model performance

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(comparison['Model'], comparison['Validation_RMSLE'])

plt.title("Model Comparison")
plt.xlabel("Models")
plt.ylabel("Validation_RMSLE")

for i, v in enumerate(comparison['Validation_RMSLE']):
    plt.text(i, v+0.002, f"{v:.4f}", ha='center')

plt.show()

In [ ]:
custom_sample = X_train.iloc[[0]].copy()

custom_sample['AssetAge'] = 5.0
custom_sample['OperationalHoursMeter'] = 2500.0
custom_sample['ManufactureYear'] = 2010

custom_pred_log = model.predict(custom_sample)
custom_pred_price = np.expm1(custom_pred_log)[0]

print(f"Predicted selling price for your custom machine configuration: ${custom_pred_price:,.2f}")

In [ ]:
import pickle
pickle_model_path = 'model.pkl'
with open(pickle_model_path, 'wb') as file:
  pickle.dump(model, file)

In [ ]:
with open(pickle_model_path, 'rb') as file:
    model = pickle.load(file)

In [ ]:
custom_sample = X_train.iloc[[0]].copy()

custom_sample['AssetAge'] = 1.0
custom_sample['OperationalHoursMeter'] = 2000.0
custom_sample['ManufactureYear'] = 2010

custom_pred_log = model.predict(custom_sample)
custom_pred_price = np.expm1(custom_pred_log)[0]

print(f"Predicted selling price for your custom machine configuration: ${custom_pred_price:,.2f}")